# Decision Tree scalability — UCI Covertype

Thí nghiệm bổ sung trên 581.012 mẫu, 54 features và 7 lớp. Notebook so sánh Decision Tree baseline với một cấu hình regularized, dùng hold-out test và 5-fold cross-validation.

**Kaggle accelerator:** CPU là lựa chọn phù hợp vì `sklearn.tree.DecisionTreeClassifier` không dùng GPU. Nếu bật GPU, notebook vẫn ghi tên GPU/VRAM/driver để mô tả môi trường, đồng thời ghi rõ model chạy CPU.

## 1. Import, cấu hình và phần cứng

In [ ]:
import json
import os
import platform
import subprocess
import time
from datetime import UTC, datetime
from pathlib import Path
from shutil import which
from zipfile import ZIP_DEFLATED, ZipFile

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None
    def display(value):
        print(value)

from sklearn.datasets import fetch_covtype
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

PIPELINE_STARTED = time.perf_counter()
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
CV_JOBS = 2
EXPERIMENT_ID = "dt_covertype_scalability"
TARGET = "cover_type"
QUICK_MODE = os.getenv("COVERTYPE_QUICK_MODE", "0") == "1"

def detect_hardware():
    info = {
        "environment": "kaggle" if Path("/kaggle/working").exists() else "local",
        "platform": platform.platform(),
        "processor": platform.processor() or platform.machine(),
        "logical_cpu_count": os.cpu_count(),
        "gpu_available": False,
        "gpus": [],
        "model_compute_device": "cpu",
    }
    nvidia_smi = which("nvidia-smi")
    if nvidia_smi:
        query = subprocess.run(
            [nvidia_smi, "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=False,
        )
        if query.returncode == 0 and query.stdout.strip():
            info["gpu_available"] = True
            for line in query.stdout.strip().splitlines():
                name, memory_mb, driver = [part.strip() for part in line.split(",", maxsplit=2)]
                info["gpus"].append(
                    {"name": name, "memory_mb": int(float(memory_mb)), "driver_version": driver}
                )
    return info

HARDWARE_INFO = detect_hardware()
IS_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
FIGURES_DIR = OUTPUT_ROOT / "figures"
RESULTS_DIR = OUTPUT_ROOT / "results"
MODELS_DIR = OUTPUT_ROOT / "models"
for directory in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
print({"python": platform.python_version(), "sklearn": sklearn.__version__, "kaggle": IS_KAGGLE})
print("Hardware:", HARDWARE_INFO)
print("Quick mode:", QUICK_MODE)

## 2. Đọc dữ liệu

Notebook ưu tiên `covertype.csv`/`covtype.csv` trong Kaggle Input hoặc project. Nếu không có, `fetch_covtype` sẽ tải bản UCI chính thức; lần chạy Kaggle đầu tiên cần bật Internet.

In [ ]:
data_started = time.perf_counter()
SEARCH_ROOTS = [Path.cwd(), Path.cwd().parent]
if Path("/kaggle/input").exists():
    SEARCH_ROOTS.insert(0, Path("/kaggle/input"))

def find_covertype_csv():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for name in ("covertype.csv", "covtype.csv"):
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    return None

def normalize_target(frame):
    target_candidates = {"cover_type", "covertype", "cover type", "target"}
    for column in frame.columns:
        if str(column).strip().lower() in target_candidates:
            return frame.rename(columns={column: TARGET})
    raise ValueError("CSV must contain a Cover_Type/cover_type/target column")

data_path = find_covertype_csv()
if data_path is not None:
    covertype_df = normalize_target(pd.read_csv(data_path))
    data_source = str(data_path)
else:
    bundle = fetch_covtype(
        as_frame=True, data_home=OUTPUT_ROOT / ".cache" / "scikit_learn_data"
    )
    covertype_df = bundle.data.copy()
    covertype_df[TARGET] = bundle.target.astype("int64").to_numpy()
    data_source = "sklearn.datasets.fetch_covtype() / UCI Covertype"

covertype_df[TARGET] = pd.to_numeric(covertype_df[TARGET], errors="raise").astype("int64")
assert covertype_df.shape[1] == 55
assert covertype_df[TARGET].nunique() == 7
assert not covertype_df.isna().any().any()

full_shape = covertype_df.shape
if QUICK_MODE and len(covertype_df) > 50_000:
    _, covertype_df = train_test_split(
        covertype_df, test_size=50_000, random_state=RANDOM_STATE, stratify=covertype_df[TARGET]
    )
    covertype_df = covertype_df.reset_index(drop=True)

FEATURES = [column for column in covertype_df.columns if column != TARGET]
dataset_memory_mb = covertype_df.memory_usage(deep=True).sum() / 1024**2
data_loading_seconds = time.perf_counter() - data_started
print("Source:", data_source)
print("Full shape / active shape:", full_shape, covertype_df.shape)
print(f"DataFrame memory: {dataset_memory_mb:.1f} MB")
display(covertype_df.head())

## 3. Phân bố lớp

In [ ]:
class_counts = covertype_df[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 5))
class_counts.plot.bar(ax=ax, color="#15803D")
ax.set(title="Covertype - class distribution", xlabel="Cover type", ylabel="Samples")
fig.tight_layout()
class_path = FIGURES_DIR / f"{EXPERIMENT_ID}__class_distribution.png"
fig.savefig(class_path, dpi=200, bbox_inches="tight")
plt.show()
display(class_counts.rename("count").to_frame().T)
print("Class imbalance ratio:", round(class_counts.max() / class_counts.min(), 3))

## 4. Hold-out split và hai cấu hình cây

Cây regularized được đặt trước khi xem test set (`max_depth=20`, `min_samples_leaf=5`) để tạo đối chứng về generalization và complexity; đây không phải cấu hình tối ưu cuối cùng.

In [ ]:
X = covertype_df[FEATURES]
y = covertype_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train/Test:", X_train.shape, X_test.shape)

model_specs = {
    "baseline": {"criterion": "gini", "random_state": RANDOM_STATE},
    "regularized": {
        "criterion": "gini", "max_depth": 20, "min_samples_leaf": 5,
        "random_state": RANDOM_STATE,
    },
}
fitted_models = {}
predictions = {}
holdout_records = []
for model_name, parameters in model_specs.items():
    model = DecisionTreeClassifier(**parameters)
    fit_started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - fit_started

    predict_started = time.perf_counter()
    train_prediction = model.predict(X_train)
    test_prediction = model.predict(X_test)
    predict_seconds = time.perf_counter() - predict_started

    train_accuracy = accuracy_score(y_train, train_prediction)
    test_accuracy = accuracy_score(y_test, test_prediction)
    holdout_records.append(
        {
            "model": model_name,
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "error_rate": 1 - test_accuracy,
            "precision_macro": precision_score(y_test, test_prediction, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, test_prediction, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, test_prediction, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_test, test_prediction, average="weighted", zero_division=0),
            "generalization_gap": train_accuracy - test_accuracy,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "tree_depth": model.get_depth(),
            "leaf_count": model.get_n_leaves(),
        }
    )
    fitted_models[model_name] = model
    predictions[model_name] = test_prediction

holdout_df = pd.DataFrame(holdout_records).set_index("model")
display(holdout_df.round(4))

## 5. 5-fold cross-validation

Theo cách trình bày của report tham khảo, notebook báo mean ± standard deviation và runtime qua các fold. CV chỉ chạy trên train split; test set được giữ lại cho đánh giá cuối.

In [ ]:
active_cv_folds = 2 if QUICK_MODE else CV_FOLDS
cv = StratifiedKFold(n_splits=active_cv_folds, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "f1_macro": "f1_macro"}
cv_records = []
for model_name, parameters in model_specs.items():
    cv_output = cross_validate(
        DecisionTreeClassifier(**parameters), X_train, y_train, cv=cv, scoring=scoring,
        return_train_score=True, n_jobs=CV_JOBS,
    )
    cv_records.append(
        {
            "model": model_name,
            "cv_folds": active_cv_folds,
            "train_accuracy_mean": cv_output["train_accuracy"].mean(),
            "train_accuracy_std": cv_output["train_accuracy"].std(),
            "test_accuracy_mean": cv_output["test_accuracy"].mean(),
            "test_accuracy_std": cv_output["test_accuracy"].std(),
            "test_f1_macro_mean": cv_output["test_f1_macro"].mean(),
            "test_f1_macro_std": cv_output["test_f1_macro"].std(),
            "fit_seconds_mean": cv_output["fit_time"].mean(),
            "fit_seconds_std": cv_output["fit_time"].std(),
            "score_seconds_mean": cv_output["score_time"].mean(),
            "score_seconds_std": cv_output["score_time"].std(),
        }
    )

cv_df = pd.DataFrame(cv_records).set_index("model")
display(cv_df.round(4))
for model_name, row in cv_df.iterrows():
    print(
        f"{model_name}: accuracy={row['test_accuracy_mean']:.4f} ± {row['test_accuracy_std']:.4f}; "
        f"macro-F1={row['test_f1_macro_mean']:.4f} ± {row['test_f1_macro_std']:.4f}; "
        f"fit={row['fit_seconds_mean']:.2f} ± {row['fit_seconds_std']:.2f}s"
    )
best_model_name = cv_df["test_f1_macro_mean"].idxmax()
best_model = fitted_models[best_model_name]
best_prediction = predictions[best_model_name]
print("Selected by CV macro-F1:", best_model_name)

## 6. So sánh hiệu năng, runtime và complexity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
holdout_df[["test_accuracy", "f1_macro"]].plot.bar(
    ax=axes[0], color=["#2563EB", "#0F766E"], ylim=(0, 1)
)
axes[0].set(title="Hold-out performance", xlabel="Model", ylabel="Score")
axes[0].tick_params(axis="x", rotation=0)
holdout_df[["tree_depth", "leaf_count"]].plot.bar(
    ax=axes[1], secondary_y="leaf_count", color=["#D97706", "#7C3AED"]
)
axes[1].set(title="Tree complexity", xlabel="Model", ylabel="Depth")
axes[1].tick_params(axis="x", rotation=0)
fig.tight_layout()
comparison_path = FIGURES_DIR / f"{EXPERIMENT_ID}__model_comparison.png"
fig.savefig(comparison_path, dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_prediction,
    labels=best_model.classes_,
    cmap="Blues",
    colorbar=False,
    values_format="d",
    ax=ax,
)
ax.set_title(f"Covertype - confusion matrix ({best_model_name})")
fig.tight_layout()
confusion_path = FIGURES_DIR / f"{EXPERIMENT_ID}__confusion_matrix.png"
fig.savefig(confusion_path, dpi=200, bbox_inches="tight")
plt.show()

importance = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values().tail(20)
fig, ax = plt.subplots(figsize=(10, 7))
importance.plot.barh(ax=ax, color="#166534")
ax.set(title=f"Top feature importance ({best_model_name})", xlabel="Gini importance", ylabel="")
fig.tight_layout()
importance_path = FIGURES_DIR / f"{EXPERIMENT_ID}__feature_importance.png"
fig.savefig(importance_path, dpi=200, bbox_inches="tight")
plt.show()

## 7. Lưu bảng, model và result JSON

In [ ]:
holdout_path = RESULTS_DIR / f"{EXPERIMENT_ID}__holdout_comparison.csv"
cv_path = RESULTS_DIR / f"{EXPERIMENT_ID}__cross_validation.csv"
holdout_df.to_csv(holdout_path)
cv_df.to_csv(cv_path)

model_paths = {}
for model_name, model in fitted_models.items():
    path = MODELS_DIR / f"{EXPERIMENT_ID}__{model_name}.joblib"
    joblib.dump(model, path)
    model_paths[model_name] = path

best_holdout = holdout_df.loc[best_model_name]
result = {
    "schema_version": "1.0",
    "experiment_id": EXPERIMENT_ID,
    "dataset": "covertype",
    "model": "DecisionTreeClassifier",
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": True},
    "metrics": {
        "test_accuracy": float(best_holdout["test_accuracy"]),
        "error_rate": float(best_holdout["error_rate"]),
        "precision_macro": float(best_holdout["precision_macro"]),
        "recall_macro": float(best_holdout["recall_macro"]),
        "f1_macro": float(best_holdout["f1_macro"]),
        "generalization_gap": float(best_holdout["generalization_gap"]),
        "data_loading_seconds": float(data_loading_seconds),
        "training_seconds": float(best_holdout["fit_seconds"]),
        "fit_seconds": float(best_holdout["fit_seconds"]),
        "prediction_seconds": float(best_holdout["predict_seconds"]),
        "predict_seconds": float(best_holdout["predict_seconds"]),
        "tree_depth": int(best_holdout["tree_depth"]),
        "leaf_count": int(best_holdout["leaf_count"]),
        "dataset_memory_mb": float(dataset_memory_mb),
    },
    "hardware": HARDWARE_INFO,
    "evaluation": {
        "selection_metric": "training-only cross-validation macro-F1",
        "cv_folds": active_cv_folds,
        "best_model": best_model_name,
        "model_specs": model_specs,
        "holdout_comparison": holdout_df.reset_index().to_dict(orient="records"),
        "cross_validation": cv_df.reset_index().to_dict(orient="records"),
    },
    "artifacts": {
        "figure_paths": [str(class_path), str(comparison_path), str(confusion_path), str(importance_path)],
        "model_path": str(model_paths[best_model_name]),
    },
    "notes": "Scalability experiment; GPU may be visible but sklearn Decision Tree computes on CPU.",
    "created_at_utc": datetime.now(UTC).isoformat(),
}
result_path = RESULTS_DIR / f"{EXPERIMENT_ID}.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Best model:", best_model_name)
print("Result:", result_path)

## 8. Đóng gói output để tải về

In [ ]:
pipeline_seconds = time.perf_counter() - PIPELINE_STARTED
result["metrics"]["pipeline_seconds"] = float(pipeline_seconds)
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

archive_path = OUTPUT_ROOT / f"{EXPERIMENT_ID}__outputs.zip"
files_to_download = [
    class_path, comparison_path, confusion_path, importance_path, holdout_path, cv_path,
    result_path, *model_paths.values(),
]
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for artifact_path in files_to_download:
        archive.write(artifact_path, arcname=artifact_path.relative_to(OUTPUT_ROOT))

timings = {
    "data_loading_seconds": data_loading_seconds,
    "baseline_fit_seconds": holdout_df.loc["baseline", "fit_seconds"],
    "regularized_fit_seconds": holdout_df.loc["regularized", "fit_seconds"],
    "pipeline_seconds": pipeline_seconds,
}
display(pd.Series(timings, name="seconds").to_frame().round(4))
print(f"Created ZIP ({archive_path.stat().st_size / 1024**2:.1f} MB): {archive_path}")
if FileLink is not None:
    display(FileLink(str(archive_path)))

## Gợi ý dùng trong report

Bảng chính nên báo hold-out Accuracy, Error Rate, macro-F1, generalization gap, depth, leaves và runtime. Bảng độ ổn định báo 5-fold CV dưới dạng mean ± standard deviation. Nhấn mạnh trade-off giữa hiệu năng và độ phức tạp; không dùng test set để chọn cấu hình.